# Novelty Search (Skeleton)
For use in the experiments of the Amorphous Fortress Narrative generation

### General Pseudocode
1. Create initial population of genomes
2. Evaluate each for fitness
3. Evaluate each for novelty against archive genomes
4. Add fit and novel genomes in archive
5. Select new parents of population from novelty archive
6. Mutate children and create new population
7. (Add random back in)
8. Repeat 2-7 for n generations

(Reference: [SimSim](https://github.com/lsoros/simsim/blob/master/simsim.cpp) and [Algorithm Definition](https://algorithmafternoon.com/novelty/novelty_search_algorithm/))

### Setup

In [180]:
# imports
import random
import numpy as np
import json
import spacy
import re
from datetime import datetime
import pyinflect
from sentence_transformers import SentenceTransformer

In [181]:
# set models for NLP tasks
nlp = spacy.load("en_core_web_sm")
st_model = SentenceTransformer('all-MiniLM-L6-v2')

In [182]:
# import data
ALL_SUBJS = np.load('../bank_files/cn_full_subjects.npy', allow_pickle=True)
ALL_OBJS = np.load('../bank_files/cn_full_objects.npy', allow_pickle=True)
ALL_VERBS = np.load('../bank_files/cn_full_verbs.npy', allow_pickle=True) 
CN_GRAPH = json.load(open('../bank_files/full_word_graph_noweight.json'))

# convert to normal lists
ALL_SUBJS = [str(s) for s in ALL_SUBJS]
ALL_OBJS = [str(o) for o in ALL_OBJS]
ALL_VERBS = [str(v) for v in ALL_VERBS]

print(len(ALL_SUBJS), len(ALL_OBJS), len(ALL_VERBS))

3701 3377 1389


In [183]:
# convert all verbs to past tense
ALL_VERBS_nlp = nlp(' '.join(ALL_VERBS))
ALL_VERBS = [token._.inflect("VBD") if token._.inflect("VBD") is not None else token.text for token in ALL_VERBS_nlp ]

In [184]:
# create verbs and verb encodings
AF_VERBS = ["moved", "died", "cloned", "took", "pushed", "added", "transformed", "blocked", "chased"]
all_af_verb_encs = st_model.encode(AF_VERBS)
af_verb_enc_dict = {AF_VERBS[i]: all_af_verb_encs[i] for i in range(len(AF_VERBS))}

In [185]:
# constants
MC_MUTATE_PERC = 0.25
ENT_MUTATE_PERC = 0.1
VERB_MUTATE_PERC = 0.25
POP_SIZE = 10
NUM_GENERATIONS = 20

In [186]:
# maps a log for usage in the novelty search
class AF_Story:
    def __init__(self, log_file):
        self.log_file = log_file
        with open(log_file, 'r') as f:
            self.og_text = [line.strip() for line in f.readlines()]
        self.ent_ids = self.find_spec_ents()        # dict of entities with subject/object designation
        self.ent_reps = self.get_ent_reps()

        self.ent_order, self.mc_ent = self.find_ents()      # list of all entities in the original text
        self.verb_set = self.find_verbs()          # list of tuples of (subject entity, verb)
        self.verb_order = [v[1] for v in self.verb_set.values()]    # list of all verbs in the original text


    def find_ents(self):
        ''' Find all AF entities in the original text. '''
        af_ents = []
        for line in self.og_text:
            match = re.findall(r'(\[.\..{4}\])', line)
            if match:
                af_ents.extend(match)

        # get highest occuring entity as main character
        random.shuffle(af_ents) # shuffle to avoid biasing first entity as MC
        mc_ent = max(set(af_ents), key = af_ents.count)
        return list(set(af_ents)), mc_ent

    def find_spec_ents(self):
        ''' Identifies entities and whether they are the subject or object in the sentence. '''
        af_ents = {}
        for line in self.og_text:
            match = re.findall(r'(\[.\..{4}\])', line)
            if match:
                for i in range(len(match)):
                    if i == 0:
                        af_ents[match[i]] = 'subject'
                    elif match[i] not in af_ents:
                        af_ents[match[i]] = 'object'
                    
        return af_ents
    
    def get_ent_reps(self):
        ''' Get the symbol representations of the entities in the story '''
        return list(set([e[1] for e in self.ent_ids.keys()]))
    
    def find_verbs(self):
        ''' Find all verbs in the original text '''
        af_verbs = {}
        for i, line in enumerate(self.og_text):
            subj_ent = re.findall(r'(\[.\..{4}\])', line)
            for verb in AF_VERBS:
                if verb in line:
                    af_verbs[i] = (subj_ent[0], verb)
                    break # only take the first verb found
        return af_verbs
    


In [187]:
# test
stupid_story = AF_Story('../logs/stupid_log.txt')
print("Ent IDs:\t" + str(stupid_story.ent_ids))
print("Ent Reps:\t" + str(stupid_story.ent_reps))
print("MC Ent:\t" + str(stupid_story.mc_ent))
print("Ent Order:\t" + str(stupid_story.ent_order))
print("Verb Set:\t" + str(stupid_story.verb_set))
print("Verb Order:\t" + str(stupid_story.verb_order))

Ent IDs:	{'[%.b29d]': 'subject', '[y.c9ae]': 'subject', '[ .d474]': 'subject', '[ .6adb]': 'object', '[$.b14a]': 'subject', '[*.7b93]': 'subject', '[|.c91d]': 'subject', '[}.460f]': 'subject', '[5.6b26]': 'object', '[S.d487]': 'subject', '[!.fdaf]': 'object', '[Y.3a62]': 'object', '[o.8829]': 'object', '[;.f34e]': 'subject', '[9.547d]': 'subject'}
Ent Reps:	['Y', '}', '!', 'y', 'o', ';', '$', 'S', ' ', '*', '9', '%', '|', '5']
MC Ent:	[$.b14a]
Ent Order:	['[ .6adb]', '[5.6b26]', '[o.8829]', '[;.f34e]', '[$.b14a]', '[*.7b93]', '[%.b29d]', '[S.d487]', '[9.547d]', '[Y.3a62]', '[y.c9ae]', '[|.c91d]', '[}.460f]', '[!.fdaf]', '[ .d474]']
Verb Set:	{3: ('[%.b29d]', 'transformed'), 4: ('[ .d474]', 'cloned'), 5: ('[$.b14a]', 'moved'), 6: ('[*.7b93]', 'moved'), 7: ('[ .d474]', 'transformed'), 8: ('[}.460f]', 'blocked'), 9: ('[S.d487]', 'added'), 10: ('[$.b14a]', 'moved'), 11: ('[|.c91d]', 'pushed'), 12: ('[y.c9ae]', 'chased'), 13: ('[;.f34e]', 'added'), 14: ('[9.547d]', 'pushed')}
Verb Order:	['

In [188]:
# Object to store the genome info
class FicGenome:
    def __init__(self, story_file:str, mc:str=None, ent:dict=None, verbs:dict=None):
        '''
            mc:    str
            ent:   {og_story_id: fic_rep_ent}
            verbs: {line: fic_rep_verb} (associated with line number in original story and verb_set)
        '''
        self.story_file = story_file    
        self.mc = mc
        self.ent = ent
        self.verbs = verbs
        self.fitness = 0
        self.genome = self.make_genome()

    def clone(self):
        ''' Returns a separate copy of this object '''
        new_fic = FicGenome(self.story_file, self.mc, {k:v for k,v in self.ent.items()}, {k:v for k,v in self.verbs.items()})
        new_fic.fitness = self.fitness
        new_fic.genome = self.genome
        return new_fic

    def mutate(self, mc='random', ent='random', verbs='random'):
        ''' Mutates the FicGenome object
            mc:    'random' | 'same'
            ent:   'random' | 'assoc'
            verbs: 'random' | 'assoc'
        '''
        # TODO: mutate based on associations
        self.genome = None # reset genome


        # remake the genome based on new values
        self.genome = self.make_genome()

    def eval(self, af_story, debug=False):
        ''' Evaluates the fitness of the FicGenome object 
            Fitness is based on:
                - Interestingness
                - Semantic closeness of the verb selection to the original verbs
        '''
        
        # get encodings for verbs
        og_verbs = list(af_story.verb_set.values())
        fic_verbs = list(self.verbs.values())
        og_verb_vecs = [af_verb_enc_dict[v[1]] for v in og_verbs]
        fic_verb_vecs = [st_model.encode(v) for v in fic_verbs]

        # get cosine similarity between verb sets
        cos_sims = []
        for i in range(len(og_verb_vecs)):
            cos_sim = np.dot(og_verb_vecs[i], fic_verb_vecs[i]) / (np.linalg.norm(og_verb_vecs[i]) * np.linalg.norm(fic_verb_vecs[i]))
            cos_sims.append(cos_sim)

            if debug:
                print(f"OG Verb: {og_verbs[i][1]} | Fic Verb: {fic_verbs[i]} | Cosine Sim: {float(cos_sim):.4f}")


        # TODO: add interestingness metric

        # set the fitness
        self.fitness = float(np.mean(cos_sims))
        return self.fitness

    def make_genome(self):
        ''' Creates a representation of the genome (ents+verbs) 
            Uses a sentence embedding model from sentence-transformers to convert the genome to a vector
            for comparison with other genomes.
        '''
        all_words = list(self.ent.values()) + list(self.verbs.values())
        all_words = ' '.join(all_words)
        genome = st_model.encode(all_words)
        return genome

    def generate_story(self, af_story, out_file:str=None):
        ''' Creates a story log based on the genome '''
        new_story = []
        for i, og_line in enumerate(af_story.og_text):
            new_line = og_line
            
            # Replace entities in the original line with their representations
            for ent_id, fic_rep in self.ent.items():
                new_line = new_line.replace(ent_id, f"[{fic_rep}]")

            # Replace verbs in the original line with their representations
            if i in self.verbs:
                af_story_verb = af_story.verb_set[i]
                fic_verb = self.verbs[i]
                new_line = new_line.replace(af_story_verb[1], fic_verb)

            new_story.append(new_line)

        # Write the modified line to the output file
        if out_file is not None:
            with open(out_file, 'w') as f:
                for line in new_story:
                    f.write(line + '\n')

        return new_story
    

    def export_fic(self, af_story, out_file:str=None):
        ''' Exports the data as a JSON file to reimport later '''
        fic_data = {
            'mc': self.mc,
            'ent': self.ent,
            'verbs': self.verbs,
            'fitness': self.fitness,
            'story_file': self.story_file,
            'new_story': '\n'.join(self.generate_story(af_story)) if af_story is not None else None
        }
        if out_file is not None:
            with open(out_file, 'w') as f:
                json.dump(fic_data, f, indent=3)
        return fic_data

    def import_fic(self, dat):
        ''' Imports the data from a JSON file '''
        self.mc = dat['mc']
        self.ent = dat['ent']
        self.verbs = dat['verbs']
        self.fitness = dat['fitness']
        self.story_file = dat['story_file']
        self.genome = self.make_genome()

### Perform novelty search on a stripped log

In [189]:
def init_population(size, story, style='random'):
    ''' Initializes a population of FicGenome objects based on the given AF_Story object '''
    population = []

    if style == 'random':
        # initialize fully randomly
        for _ in range(size):
            # choose a random MC from the entities in the story
            mc = random.choice(ALL_SUBJS)
            
            # choose random entities from the entities in the story
            ent = {}
            for k,v in story.ent_ids.items():
                if k == story.mc_ent:
                    ent[story.mc_ent] = mc
                else:
                    if v == 'subject':
                        ent[k] = random.choice(ALL_SUBJS)
                    else:
                        ent[k] = random.choice(ALL_OBJS)

            # choose completely random verbs from the verbs in the story
            verbs = {}
            for k,v in story.verb_set.items():
                # pick random verb
                verbs[k] = random.choice(ALL_VERBS)
                

            fic = FicGenome(story.log_file, mc, ent, verbs)
            population.append(fic)

    # TODO: create population based on associations

    
    return population

In [190]:
# test generating a story
pop = init_population(1, stupid_story, style='random')
fic = pop[0]
print(f"\nFic: MC={fic.mc}, Ents={fic.ent}, Verbs={fic.verbs}")
print(fic.eval(stupid_story))


Fic: MC=crab, Ents={'[%.b29d]': 'parent', '[y.c9ae]': 'fruit', '[ .d474]': 'urinalysis', '[ .6adb]': 'thumbtack', '[$.b14a]': 'crab', '[*.7b93]': 'crash', '[|.c91d]': 'classroom', '[}.460f]': 'canadian', '[5.6b26]': 'subdivision', '[S.d487]': 'votive', '[!.fdaf]': 'pole', '[Y.3a62]': 'lightbulb', '[o.8829]': 'testube', '[;.f34e]': 'supplier', '[9.547d]': 'tourqouise'}, Verbs={3: 'seated', 4: 'complemented', 5: 'snapped', 6: 'united', 7: 'exited', 8: 'led', 9: 'billionaire', 10: 'hauled', 11: 'followed', 12: 'graded', 13: 'compensated', 14: 'sucked'}
0.30604586005210876


In [191]:
def is_novel(x, archive, threshold=0.5, debug=False):
    ''' Determines if a FicGenome object is novel compared to an archive of FicGenome objects '''
    if len(archive) == 0:       # nothing in the archive yet, so it's novel!
        return True

    # Get the minimum distance to any genome in the archive
    min_distance = float('inf')
    distances = []
    for a in archive:
        distance = np.linalg.norm(x.genome - a.genome)
        distances.append(distance)
        if distance < min_distance:
            min_distance = distance

    if debug:
        print(f"Distances: {distances}")

    # If the minimum distance is greater than the threshold, the genome is novel
    return min_distance >= threshold


In [192]:
def export_archive(arx, out_file:str, story=None):
    ''' Exports the archive of FicGenome objects to a JSON file '''
    archive_data = [x.export_fic(story) for x in arx]
    with open("../novelty_search_out/archive"+out_file, 'w') as f:
        json.dump(archive_data, f, indent=3)

In [193]:
def novelty_search(af_log, fit_threshold=0.5, novel_threshold=0.5, rand_perc=0.2):
    ''' Main novelty search algorithm '''

    # 0. Initialize story representation
    story = AF_Story(af_log)
    
    # 1. Initialize population and archive
    population = init_population(POP_SIZE, story)
    archive = []

    best_fitness = 0
    best_fic = None

    for gen in range(NUM_GENERATIONS):
        print(f"Generation {gen}")

        # 8. Repeat 2-7 for NUM_GENERATIONS
        for indiv in population:

            # 2. Evaluate fitness
            indiv.eval(story)

            # 3+4. Evaluate novelty against archive and add if novel and fit enough
            if is_novel(indiv, archive, novel_threshold) and indiv.fitness > fit_threshold:
                archive.append(indiv.clone())


        # print some stats
        population.sort(key=lambda x: x.fitness, reverse=True)
        fit_scores = [indiv.fitness for indiv in population]
        print(f"  Pop Fitness: max {max(fit_scores):.3f}, min {min(fit_scores):.3f}, avg {sum(fit_scores)/len(fit_scores):.3f}")
        print(f"  Archive size: {len(archive)}")

        if gen % 10 == 0:
            print("   FicGenome of best population individual:")
            print(f"     - Best MC: {population[0].mc}")
            print(f"     - Best Ents: {list(population[0].ent.values())}")
            print(f"     - Best Verbs: {set(population[0].verbs.values())}")

        
        if population[0].fitness > best_fitness:
            best_fitness = population[0].fitness
            best_fic = population[0].clone()
            print(f"  New best fitness: {best_fitness:.3f}")

        # 5. Select new parents from novelty archive
        if len(archive) > 0:
            parents = random.choices(archive, k=int(POP_SIZE*(1-rand_perc)))
        else:
            parents = random.choices(population, k=int(POP_SIZE*(1-rand_perc)))

        # 6. Mutate children from parents
        new_pop = []
        for parent in parents:
            child = parent.clone()
            child.mutate()
            new_pop.append(child)

        # 7. Add random individuals
        rand_amt = (POP_SIZE - len(new_pop))
        for _ in range(int(POP_SIZE*rand_perc)):
            randos = init_population(rand_amt, story)
            new_pop.extend(randos)

        # update population
        population = new_pop

    return archive, best_fic, story

In [194]:
arc, best_fic, story = novelty_search('../logs/stupid_log.txt', fit_threshold=0.5, novel_threshold=0.5, rand_perc=0.2)

Generation 0
  Pop Fitness: max 0.300, min 0.228, avg 0.268
  Archive size: 0
   FicGenome of best population individual:
     - Best MC: paroxetine
     - Best Ents: ['striker', 'promotion', 'inaction', 'experiment', 'paroxetine', 'role', 'tank', 'hacker', 'taxis', 'crocodile', 'stereoe', 'stapler', 'havoc', 'sampoo', 'atheist']
     - Best Verbs: {'set', 'singed', 'pilate', 'stayed', 'gifted', 'clowned', 'processed', 'had', 'wrenched', 'went', 'logged', 'sprang'}
  New best fitness: 0.300
Generation 1
  Pop Fitness: max 0.300, min 0.228, avg 0.269
  Archive size: 0
Generation 2
  Pop Fitness: max 0.302, min 0.228, avg 0.267
  Archive size: 0
  New best fitness: 0.302
Generation 3
  Pop Fitness: max 0.309, min 0.228, avg 0.273
  Archive size: 0
  New best fitness: 0.309
Generation 4
  Pop Fitness: max 0.315, min 0.248, avg 0.282
  Archive size: 0
  New best fitness: 0.315
Generation 5
  Pop Fitness: max 0.315, min 0.258, avg 0.284
  Archive size: 0
Generation 6
  Pop Fitness: max 0.30

In [195]:
# save the best fic story
t = datetime.now().strftime("[%m-%d-%Y %H%M]")
print(best_fic.export_fic(story, out_file=f'../novelty_search_out/fic_genomes/best_fic_log-{POP_SIZE}_{NUM_GENERATIONS}_{t}.json'))
best_fic.generate_story(story, out_file=f'../novelty_search_out/gen_stories/best_fic_log-{POP_SIZE}_{NUM_GENERATIONS}_{t}.txt')

{'mc': 'arsonist', 'ent': {'[%.b29d]': 'motehr', '[y.c9ae]': 'saving', '[ .d474]': 'pleasure', '[ .6adb]': 'anxiety', '[$.b14a]': 'arsonist', '[*.7b93]': 'gunshot', '[|.c91d]': 'vineyard', '[}.460f]': 'chipmunk', '[5.6b26]': 'truth', '[S.d487]': 'wwf', '[!.fdaf]': 'sustenance', '[Y.3a62]': 'applause', '[o.8829]': 'weight', '[;.f34e]': 'evangelist', '[9.547d]': 'journalist'}, 'verbs': {3: 'returna', 4: 'referenced', 5: 'kept', 6: 'compared', 7: 'obtained', 8: 'confused', 9: 'realized', 10: 'landed', 11: 'jotted', 12: 'inmate', 13: 'dinned', 14: 'smeared'}, 'fitness': 0.31541910767555237, 'story_file': '../logs/stupid_log.txt', 'new_story': '=====    FORTRESS SEED: [791231]    =====\nFortress initialized! - <0>\n>>> TIME: 2025-08-14 13:21:44 <<<\n<0> [motehr] returna into [saving]\n<1> [pleasure] referenced to [anxiety] at (44, 2)\n<2> [arsonist] kept to (65, 47)\n<3> [gunshot] compared to (65, 68)\n<5> [pleasure] obtained into [vineyard]\n<6> [chipmunk] confused by [truth]\n<7> [wwf] re

['=====    FORTRESS SEED: [791231]    =====',
 'Fortress initialized! - <0>',
 '>>> TIME: 2025-08-14 13:21:44 <<<',
 '<0> [motehr] returna into [saving]',
 '<1> [pleasure] referenced to [anxiety] at (44, 2)',
 '<2> [arsonist] kept to (65, 47)',
 '<3> [gunshot] compared to (65, 68)',
 '<5> [pleasure] obtained into [vineyard]',
 '<6> [chipmunk] confused by [truth]',
 '<7> [wwf] realized [sustenance] at (55, 7)',
 '<8> [arsonist] landed to (50, 78)',
 '<9> [vineyard] jotted [applause]',
 '<10> [saving] inmate [weight]',
 '<11> [evangelist] dinned [wwf] at (37, 88)',
 '<12> [journalist] smeared [motehr]']

In [196]:
is_novel(best_fic, arc, threshold=1, debug=True)

True

In [197]:
fic.eval(stupid_story, debug=True)

OG Verb: transformed | Fic Verb: seated | Cosine Sim: 0.2263
OG Verb: cloned | Fic Verb: complemented | Cosine Sim: 0.2626
OG Verb: moved | Fic Verb: snapped | Cosine Sim: 0.5033
OG Verb: moved | Fic Verb: united | Cosine Sim: 0.3034
OG Verb: transformed | Fic Verb: exited | Cosine Sim: 0.2458
OG Verb: blocked | Fic Verb: led | Cosine Sim: 0.2127
OG Verb: added | Fic Verb: billionaire | Cosine Sim: 0.1069
OG Verb: moved | Fic Verb: hauled | Cosine Sim: 0.4977
OG Verb: pushed | Fic Verb: followed | Cosine Sim: 0.5266
OG Verb: chased | Fic Verb: graded | Cosine Sim: 0.2200
OG Verb: added | Fic Verb: compensated | Cosine Sim: 0.2405
OG Verb: pushed | Fic Verb: sucked | Cosine Sim: 0.3270


0.30604586005210876

In [ ]:
# # test the fitness evaluation function with the word semantic similarity
# def test_eval(og_verbs, fic_verbs):
#     ''' Evaluates the fitness of the FicGenome object 
#         Fitness is based on:
#             - Interestingness
#             - Semantic closeness of the verb selection to the original verbs
#     '''
    
#     # get encodings for verbs
#     og_verb_vecs = [af_verb_enc_dict[v] for v in og_verbs]
#     all_verbs_vecs = nlp(' '.join(fic_verbs))
#     fic_verb_vecs = [all_verbs_vecs[i] for i in range(len(og_verb_vecs))]

#     # get cosine similarity between verb sets
#     cos_sims = []
#     for i in range(len(og_verb_vecs)):
#         cos_sim = og_verb_vecs[i].similarity(fic_verb_vecs[i])
#         cos_sims.append(cos_sim)

#         print(f"OG Verb: {og_verbs[i]} | Fic Verb: {fic_verbs[i]} | Cosine Sim: {float(cos_sim):.4f}")

#     return float(np.mean(cos_sims))

# test_eval(['moved', 'transformed', 'took'], ['drove', 'altered', 'looked'])

OG Verb: moved | Fic Verb: drove | Cosine Sim: 0.3330
OG Verb: transformed | Fic Verb: altered | Cosine Sim: 0.2628
OG Verb: took | Fic Verb: looked | Cosine Sim: 1.0000


0.531936913728714

In [ ]:
# # create verbs and verb encodings with sentence transformers 
# much more accurate!

# def alt_eval(og_verbs, fic_verbs):
#     ''' Alternative evaluation function using embeddings from sentence transformers '''
#     og_verb_vecs = st_model.encode(og_verbs)
#     fic_verb_vecs = st_model.encode(fic_verbs)
#     cos_sims = []
#     for i in range(len(og_verb_vecs)):
#         cos_sim = np.dot(og_verb_vecs[i], fic_verb_vecs[i]) / (np.linalg.norm(og_verb_vecs[i]) * np.linalg.norm(fic_verb_vecs[i]))
#         cos_sims.append(cos_sim)
#         print(f"OG Verb: {og_verbs[i]} | Fic Verb: {fic_verbs[i]} | Cosine Sim: {float(cos_sim):.4f}")

#     return float(np.mean(cos_sims))

# alt_eval(['moved', 'transformed', 'took'], ['drove', 'transtitioned', 'took'])


OG Verb: moved | Fic Verb: drove | Cosine Sim: 0.4614
OG Verb: transformed | Fic Verb: transtitioned | Cosine Sim: 0.6190
OG Verb: took | Fic Verb: took | Cosine Sim: 1.0000


0.6934957504272461